# Neural Network V8

In V6 and V7 I only used 1000 training samples and 200 validation samples. For this experiment I wanted to see how much more training data helps, so I tested 1000, 2000, 4000, 8000 and all the training samples. I also switched to the full validation sets (4000 image and 2400 sentiment) because 200 samples was too small to tell the settings apart.

First, I imported the libraries and functions I'll need.

In [1]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import torch
from torch import nn

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_loader import load_task, split_train_val, standardize
from src.neural_net import (
    ModelConfig,
    evaluate,
    parameter_count,
    predict_probabilities,
    train_model,
)

epochs_by_task = {'image': 30, 'sentiment': 5}
device = torch.device('cpu')
config = ModelConfig(
    hidden_sizes=(64,),
    optimizer='adam',
    learning_rate=0.001,
    batch_size=32,
    activation='relu',
)

I kept the ReLU settings that were selected in V7. Before changing anything, I reran the V7 setup with 1000 training and 200 validation samples to make sure I still get the same accuracies.

In [2]:
def train_and_score(X_train, y_train, X_val, y_val, task):
    X_train, X_val = standardize(X_train, X_val)
    output_size = 50 if task == 'image' else 3
    started = time.perf_counter()
    result = train_model(
        X_train,
        y_train,
        output_size,
        config,
        epochs_by_task[task],
        seed=42,
        device=device,
        validation_features=X_val,
        validation_labels=y_val,
        early_stopping=False,
    )
    training_time = time.perf_counter() - started
    _, training_accuracy = evaluate(
        result.model, X_train, y_train, config.batch_size, device, nn.CrossEntropyLoss()
    )
    prediction_started = time.perf_counter()
    probabilities = predict_probabilities(result.model, X_val, config.batch_size, device)
    prediction_time = time.perf_counter() - prediction_started
    predictions = probabilities.argmax(axis=1)
    return {
        'result': result,
        'predictions': predictions,
        'validation_accuracy': float(np.mean(predictions == y_val)),
        'training_accuracy': float(training_accuracy),
        'training_time': training_time,
        'prediction_time': prediction_time,
    }


splits = {}
for task in ('image', 'sentiment'):
    X, y, _, _ = load_task(task, data_dir=ROOT / 'data')
    splits[task] = split_train_val(X, y, val_fraction=0.2, seed=42)
    X_train, X_val, y_train, y_val = splits[task]
    check = train_and_score(X_train[:1000], y_train[:1000], X_val[:200], y_val[:200], task)
    print(f"{task}: 1000/200 validation accuracy {check['validation_accuracy']:.3f}")

image: 1000/200 validation accuracy 0.150
sentiment: 1000/200 validation accuracy 0.610


Next I wrote the loop for the training sizes. For each size I fit the standardization on that training subset only, and every model is checked on the same full validation set.

In [3]:
training_sizes = (1000, 2000, 4000, 8000, None)


def run_size_sweep(task):
    X_train, X_val, y_train, y_val = splits[task]
    rows = []
    predictions_by_size = {}
    for size in training_sizes:
        n = len(X_train) if size is None else size
        run = train_and_score(X_train[:n], y_train[:n], X_val, y_val, task)
        result = run['result']
        rows.append({
            'version': 8,
            'task': task,
            'training_samples': n,
            'validation_samples': len(X_val),
            'epochs': epochs_by_task[task],
            'best_epoch': result.best_epoch,
            'config': (
                f'hidden_sizes=[64]; activation=relu; optimizer=adam; '
                f'learning_rate=0.001; batch_size=32; normalized=true; '
                f'training_samples={n}; validation_samples={len(X_val)}'
            ),
            'parameters': parameter_count(result.model),
            'training_time_seconds': run['training_time'],
            'prediction_time_seconds': run['prediction_time'],
            'best_val_accuracy': run['validation_accuracy'],
            'last_epoch_val_accuracy': float(result.history[-1]['val_accuracy']),
            'final_train_accuracy': run['training_accuracy'],
            'train_val_gap': run['training_accuracy'] - run['validation_accuracy'],
        })
        predictions_by_size[n] = run['predictions']

    # Highest validation accuracy wins; a tie goes to the smaller training set.
    best = max(rows, key=lambda row: (row['best_val_accuracy'], -row['training_samples']))
    return {
        'rows': rows,
        'best': best,
        'actual_labels': y_val,
        'predictions': predictions_by_size[best['training_samples']],
    }


columns = ['training_samples', 'best_epoch', 'final_train_accuracy',
           'best_val_accuracy', 'last_epoch_val_accuracy', 'training_time_seconds']

I started with the image data.

In [4]:
image_results = run_size_sweep('image')
print(pd.DataFrame(image_results['rows'])[columns].to_string(index=False))

 training_samples  best_epoch  final_train_accuracy  best_val_accuracy  last_epoch_val_accuracy  training_time_seconds
             1000          27              1.000000            0.13675                  0.13650               2.265515
             2000           8              0.963000            0.17575                  0.16500               3.211850
             4000           7              0.892250            0.21475                  0.19525               5.259778
             8000           5              0.696875            0.25500                  0.21800               9.441585
            16000           5              0.616250            0.29525                  0.25475              17.577045


Then I did the same test with the sentiment data.

In [5]:
sentiment_results = run_size_sweep('sentiment')
print(pd.DataFrame(sentiment_results['rows'])[columns].to_string(index=False))

 training_samples  best_epoch  final_train_accuracy  best_val_accuracy  last_epoch_val_accuracy  training_time_seconds
             1000           5                0.9680           0.625833                 0.625833               0.286195
             2000           3                0.8595           0.645417                 0.627083               0.457509
             4000           3                0.8265           0.653750                 0.642917               0.803991
             8000           1                0.7155           0.672083                 0.650000               1.552405
             9600           2                0.7575           0.673333                 0.644583               1.849001


Finally, I saved all the results and the validation predictions from the best training size.

In [6]:
for task, task_results in (('image', image_results), ('sentiment', sentiment_results)):
    pd.DataFrame(task_results['rows']).to_csv(
        ROOT / 'results' / task / 'nn_v8_summary.csv', index=False
    )
    validation = pd.DataFrame({
        'validation_index': np.arange(len(task_results['actual_labels'])),
        'actual_label': task_results['actual_labels'],
        'predicted_label': task_results['predictions'],
    })
    validation.to_csv(ROOT / 'results' / task / 'nn_v8_validation.csv', index=False)
    saved_accuracy = (validation['actual_label'] == validation['predicted_label']).mean()
    best = task_results['best']
    print(
        f"{task}: best with {best['training_samples']} training samples, "
        f"validation accuracy {best['best_val_accuracy']:.4f} "
        f"(saved file gives {saved_accuracy:.4f}, {len(validation)} rows)"
    )

image: best with 16000 training samples, validation accuracy 0.2953 (saved file gives 0.2953, 4000 rows)
sentiment: best with 9600 training samples, validation accuracy 0.6733 (saved file gives 0.6733, 2400 rows)


Accuracy went up with every bigger training set on both datasets. For image it went from 0.137 with 1000 samples to 0.295 with all 16000, and the training accuracy dropped from 1.0 to 0.62, so the model was overfitting a lot less. The best epoch also moved much earlier (27 down to 5), and with all the data the last epoch scored 0.255 instead of 0.295, so image starts overfitting after about 5 epochs.

For sentiment the jump was smaller, from 0.626 to 0.673, and most of it came by 8000 samples. The best epoch was only 1 or 2 with the larger sets, so the 5 epochs I carried over from V7 are more than it needs.

The image accuracy is still well below what the bigger [1024, 1024] network got in V4 (0.389), so one hidden layer of 64 neurons seems too small for 50 classes once there is enough data.